In [1]:
import pandas as pd
import numpy as np
import random

# GÉNÉRATION DE BASE
categories = ['Tourisme', 'Culture', 'Tech', 'fitness', 'music', 'danse']

actions_par_categorie = {
    'Tourisme': ['clic_article_voyage', 'recherche_hotel', 'vue_video_plage'],
    'Culture': ['lecture_article_histoire', 'clic_expo_musee', 'vue_docu_art'],
    'Tech': ['vue_tuto_python', 'telechargement_app', 'clic_actu_ia'],
    'fitness': ['achat_produit_sport', 'vue_tuto_exercice', 'inscription_salle_sport'],
    'music': ['ecoute_chanson', 'achat_album', 'recherche_artiste'],
    'danse': ['vue_video_danse', 'inscription_cours_danse', 'partage_performance']
}

donnee = []

for i in range(100):
    interet = random.choice(categories)
    logs = random.choices(actions_par_categorie[interet], k=random.randint(2, 3))

    donnee.append({
        'NOM': f"User_{i}",
        'AGE': random.randint(15, 65),
        'Centre_interet': interet,
        'activity_log': logs
    })

#  IMPORTANT : création de df (ligne clé)
df = pd.DataFrame(donnee)

# --- 2. INJECTION DE BUGS (dataset sale) ---

# valeurs manquantes
df.loc[df.sample(frac=0.05).index, 'AGE'] = np.nan
df.loc[df.sample(frac=0.05).index, 'Centre_interet'] = np.nan

# valeurs aberrantes
df.loc[2, 'AGE'] = -10
df.loc[15, 'AGE'] = 250

# doublons
doublons = df.head(3)
df = pd.concat([df, doublons], ignore_index=True)

# mélange
df = df.sample(frac=1).reset_index(drop=True)

# --- 3. CHECK FINAL ---
print("Dataset prêt ")
print(df.shape)
print(df.head(10))

Dataset prêt 
(103, 4)
       NOM   AGE Centre_interet  \
0  User_68  61.0        fitness   
1  User_26  37.0        Culture   
2  User_50  39.0           Tech   
3  User_86  36.0           Tech   
4   User_0  16.0        Culture   
5  User_65  41.0        Culture   
6  User_70  32.0          danse   
7  User_20  17.0          music   
8  User_14  63.0        Culture   
9  User_37  19.0          danse   

                                        activity_log  
0  [inscription_salle_sport, inscription_salle_sp...  
1                    [clic_expo_musee, vue_docu_art]  
2              [telechargement_app, vue_tuto_python]  
3  [telechargement_app, telechargement_app, telec...  
4   [clic_expo_musee, vue_docu_art, clic_expo_musee]  
5  [vue_docu_art, lecture_article_histoire, lectu...  
6         [partage_performance, partage_performance]  
7             [recherche_artiste, recherche_artiste]  
8  [lecture_article_histoire, lecture_article_his...  
9  [partage_performance, inscription_cour

In [ ]:
#Exploration initiale
print(df.shape)
print(df.dtypes)
print(df['AGE'].describe())

(103, 4)
NOM                object
AGE               float64
Centre_interet     object
activity_log       object
dtype: object
count     99.000000
mean      38.353535
std       26.803256
min      -10.000000
25%       22.000000
50%       37.000000
75%       50.500000
max      250.000000
Name: AGE, dtype: float64


In [ ]:
#Suppression des doublons

# Convert the 'activity_log' column to a hashable type (string) to avoid TypeError
df['activity_log'] = df['activity_log'].apply(lambda x: str(x) if isinstance(x, list) else x)

nb_doublons = df.duplicated().sum()
print(f"Doublons : {nb_doublons}")

df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape après nettoyage : {df.shape}")

Doublons : 3
Shape après nettoyage : (100, 4)


In [ ]:
#Valeurs manquantes (NaN)
print(df.isnull().sum())

mediane_age = df['AGE'].median()
df['AGE'] = df['AGE'].fillna(mediane_age)

mode_interet = df['Centre_interet'].mode()[0]
df['Centre_interet'] = df['Centre_interet'].fillna(mode_interet)

print("NaN restants :", df.isnull().sum().sum())

NOM               0
AGE               4
Centre_interet    5
activity_log      0
dtype: int64
NaN restants : 0


In [ ]:
#Valeurs aberrantes (Outliers)

masque_outlier = (df['AGE'] < 15) | (df['AGE'] > 65)
print(df[masque_outlier][['NOM', 'AGE']])

mediane = df.loc[~masque_outlier, 'AGE'].median()
df.loc[masque_outlier, 'AGE'] = mediane

# Alternative : df['AGE'] = df['AGE'].clip(15, 65)

        NOM    AGE
16   User_2  -10.0
84  User_15  250.0


In [ ]:
#Nettoyage des catégories
print(df['Centre_interet'].unique())  # repérer les espaces

df['Centre_interet'] = (
    df['Centre_interet']
    .str.strip()
    .str.capitalize()
)

print(df['Centre_interet'].value_counts())

['Tech' 'fitness' 'Culture' 'Tourisme' 'danse' 'music']
Centre_interet
Tech        27
Culture     16
Tourisme    16
Fitness     14
Danse       14
Music       13
Name: count, dtype: int64


In [ ]:
#Typage & Encodage
from sklearn.preprocessing import LabelEncoder

df['AGE'] = df['AGE'].astype(int)

le = LabelEncoder()
df['Centre_interet_enc'] = le.fit_transform(df['Centre_interet'])

df['nb_actions'] = df['activity_log'].apply(len)

print(df.dtypes)

NOM                   object
AGE                    int64
Centre_interet        object
activity_log          object
Centre_interet_enc     int64
nb_actions             int64
dtype: object


In [ ]:
#Export final
print(f"Shape final : {df.shape}")
print(df.describe())

df.to_csv('dataset_propre.csv', index=False)
print("dataset_propre.csv sauvegardé !")

Shape final : (100, 6)
              AGE  Centre_interet_enc  nb_actions
count  100.000000           100.00000  100.000000
mean    37.400000             2.69000   51.510000
std     14.340027             1.72735   11.192074
min     15.000000             0.00000   30.000000
25%     23.750000             1.00000   44.000000
50%     37.000000             3.00000   51.000000
75%     49.250000             4.00000   60.000000
max     65.000000             5.00000   75.000000
dataset_propre.csv sauvegardé !
